# Practical 4: Regularization Techniques for Deep Learning

**Problem Statement:** Implement L1/L2 Regularization and Dropout to reduce overfitting in Housing Price Prediction.

**Activities:**
1. Train baseline model
2. Apply L1/L2 regularization
3. Compare performance with Dropout

**Dataset:** California Housing dataset — numeric housing features, continuous price target.

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

data = fetch_california_housing(as_frame=True)
df = data.frame
df.head()

## 2. Data Preprocessing

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 3. Baseline Model (No Regularization)

A wide, unregularized network is used deliberately so overfitting is visible in the training curves.

In [ ]:
def build_baseline():
    return keras.Sequential([
        keras.layers.Input(shape=(X_train.shape[1],)),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(1)
    ])

baseline_model = build_baseline()
baseline_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_baseline = baseline_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0
)

print("Baseline training complete.")

## 4. L1/L2 Regularization

L1 and L2 penalties are added to the Dense layer weights, discouraging large weight values and reducing model complexity.

In [ ]:
def build_l1_l2():
    return keras.Sequential([
        keras.layers.Input(shape=(X_train.shape[1],)),
        keras.layers.Dense(256, activation='relu',
                            kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4)),
        keras.layers.Dense(256, activation='relu',
                            kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4)),
        keras.layers.Dense(128, activation='relu',
                            kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4)),
        keras.layers.Dense(1)
    ])

l1_l2_model = build_l1_l2()
l1_l2_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_l1_l2 = l1_l2_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0
)

print("L1/L2 regularized training complete.")

## 5. Dropout Regularization

Dropout randomly deactivates a fraction of units during training, preventing co-adaptation between neurons.

In [ ]:
def build_dropout():
    return keras.Sequential([
        keras.layers.Input(shape=(X_train.shape[1],)),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(1)
    ])

dropout_model = build_dropout()
dropout_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_dropout = dropout_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0
)

print("Dropout training complete.")

## 6. Performance Comparison

In [ ]:
def evaluate(model, name):
    y_pred = model.predict(X_test, verbose=0).flatten()
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return {'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse}

results = pd.DataFrame([
    evaluate(baseline_model, 'Baseline'),
    evaluate(l1_l2_model, 'L1/L2 Regularized'),
    evaluate(dropout_model, 'Dropout')
])

results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(history_baseline.history['loss'], '--', color='tab:blue', label='Baseline Train')
ax.plot(history_baseline.history['val_loss'], color='tab:blue', label='Baseline Val')

ax.plot(history_l1_l2.history['loss'], '--', color='tab:orange', label='L1/L2 Train')
ax.plot(history_l1_l2.history['val_loss'], color='tab:orange', label='L1/L2 Val')

ax.plot(history_dropout.history['loss'], '--', color='tab:green', label='Dropout Train')
ax.plot(history_dropout.history['val_loss'], color='tab:green', label='Dropout Val')

ax.set_title('Training vs Validation Loss Across Regularization Methods')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()
plt.show()

## Conclusion

In this practical, we:
- Trained a baseline network prone to overfitting
- Applied L1/L2 weight regularization
- Applied Dropout regularization
- Compared all three using test MAE, MSE, RMSE and training/validation loss curves